# Practical 5B — CIFAR-10 CNN Filters and Feature Maps

## Aim
Train a basic CNN, visualize convolutional filters and shallow/deep feature maps, and analyze overfitting and gradient challenges.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Load CIFAR-10 and train a small CNN

In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
np.random.seed(42); tf.random.set_seed(42)
(x_train,y_train),(x_test,y_test)=keras.datasets.cifar10.load_data()
x_train=x_train.astype("float32")/255.; x_test=x_test.astype("float32")/255.
model=keras.Sequential([
 layers.Input((32,32,3)),layers.Conv2D(32,3,padding="same",activation="relu",name="conv_shallow"),
 layers.MaxPooling2D(),layers.Conv2D(64,3,padding="same",activation="relu",name="conv_deep"),
 layers.MaxPooling2D(),layers.Flatten(),layers.Dense(128,activation="relu"),
 layers.Dense(10,activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
history=model.fit(x_train[:20000],y_train[:20000],validation_split=.1,epochs=5,batch_size=128,verbose=1)
print("Test subset:",model.evaluate(x_test[:3000],y_test[:3000],verbose=0))

## 2. Visualize first-layer convolutional filters

In [ ]:
weights,bias=model.get_layer("conv_shallow").get_weights()
fig,axes=plt.subplots(4,8,figsize=(12,6))
for i,ax in enumerate(axes.flat):
    filt=weights[:,:,:,i]
    filt=(filt-filt.min())/(filt.max()-filt.min()+1e-8)
    ax.imshow(filt); ax.axis("off")
plt.suptitle("Learned shallow convolution filters"); plt.show()

## 3. Visualize shallow and deep feature maps

In [ ]:
sample=x_test[:1]
extractor=keras.Model(model.input,[model.get_layer("conv_shallow").output,model.get_layer("conv_deep").output])
shallow,deep=extractor.predict(sample,verbose=0)
def show_maps(maps,title,n=16):
    fig,axes=plt.subplots(4,4,figsize=(8,8))
    for i,ax in enumerate(axes.flat):
        ax.imshow(maps[0,:,:,i],cmap="viridis"); ax.axis("off")
    plt.suptitle(title); plt.show()
show_maps(shallow,"Shallow feature maps")
show_maps(deep,"Deeper feature maps")
plt.imshow(sample[0]); plt.title("Input image"); plt.axis("off"); plt.show()

## 4. Analyze overfitting and gradient concerns

In [ ]:
plt.plot(history.history["loss"],label="train")
plt.plot(history.history["val_loss"],label="validation")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Overfitting diagnostic"); plt.legend(); plt.show()

## 5. Conclusion\nShallow filters often respond to simple visual patterns; deeper maps combine features into more complex representations. A widening train/validation loss gap can indicate overfitting. Deep networks can also face unstable gradients; residual connections, normalization and suitable initialization can help.